# Lab 03: Your Thumb-Sized Window on the World

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-03.ipynb)

**What you will do:** measure how many characters you can read to each side of a fixated word, compare that span with the book's reported range, and test a simulated "foveated rendering" scheme that mimics your own fovea-centred strategy.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 03 you fixed your gaze on your own thumbnail at arm's length and noticed how little
detail you could make out elsewhere in the room, then found how far your wiggling fingers could
move to the side before becoming hard to see, and finally counted how many characters you could
read to each side of a fixated word without moving your eyes.

## Record your results

Step 5 asks how many characters you can read on each side of a fixated word without moving
your eyes. Time yourself on at least three separate lines of text (a book, a screen, a printed
page), fixating on the middle word each time and counting outward until letters blur.

In [ ]:
# example data: replace with your own counts, in characters, for three separate readings
peripheral_reading = pd.DataFrame({
    "trial": ["line 1", "line 2", "line 3"],
    "chars_left": [6, 5, 7],     # example data
    "chars_right": [7, 6, 8],    # example data
})
peripheral_reading["total_span"] = peripheral_reading["chars_left"] + peripheral_reading["chars_right"]
display(peripheral_reading)
peripheral_reading[["chars_left", "chars_right"]].plot.bar(figsize=(6, 3))
plt.ylabel("characters read without moving the eyes"); plt.tight_layout(); plt.show()

## Compare

Lab 03 reports that most people can read "about 4 to 8 characters in each direction" from a
fixated word without moving their eyes.

In [ ]:
lo, hi = 4, 8
for side in ["chars_left", "chars_right"]:
    mean_val = peripheral_reading[side].mean()
    in_range = lo <= mean_val <= hi
    print(f"{side}: mean {mean_val:.1f} characters "
          f"({'within' if in_range else 'outside'} the book's 4-8 range)")

## The AI side

Your fovea covers about 2 degrees of your visual field, yet you experience the whole scene as
sharp, because the brain invests almost all of its resolution at the point of fixation and
predicts the rest. **Foveated rendering** applies exactly this idea to save computation: the
book reports that some VR systems cut rendering work by "as much as 90%" by rendering sharply
only where the eye is looking. Below, we simulate foveated rendering on a synthetic image:
full detail at a chosen fixation point, increasingly blurred with distance from it -- like an
eye-tracked headset, except that here you can compare against the true, unblurred image
(something a VR user never gets to do with their own periphery).

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
from PIL import Image, ImageDraw, ImageFilter

# A synthetic scene with fine detail everywhere, so blur is easy to see
rng = np.random.default_rng(2026)
img = Image.new("RGB", (320, 320), (250, 250, 245))
draw = ImageDraw.Draw(img)
for _ in range(400):
    x, y = rng.integers(0, 320, 2)
    r = rng.integers(2, 6)
    colour = tuple(int(c) for c in rng.integers(0, 255, 3))
    draw.ellipse([x - r, y - r, x + r, y + r], fill=colour)

fixation = (160, 160)
blur_rings = [(60, 4), (120, 10), (200, 18)]  # (radius from fixation, blur strength)
yy, xx = np.mgrid[0:320, 0:320]
dist = np.sqrt((xx - fixation[0]) ** 2 + (yy - fixation[1]) ** 2)

foveated = np.array(img).astype(float)
for radius, blur in blur_rings:
    blurred = np.array(img.filter(ImageFilter.GaussianBlur(blur))).astype(float)
    mask = (dist >= radius)[..., None]
    foveated = np.where(mask, blurred, foveated)
foveated = foveated.astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img); axes[0].set_title("True image (every pixel sharp)", fontsize=9)
axes[1].imshow(foveated); axes[1].set_title("Foveated: sharp only near fixation", fontsize=9)
for ax in axes:
    ax.scatter(*fixation, marker="+", color="red", s=120)
    ax.axis("off")
plt.tight_layout(); plt.show()

reduced_fraction = (dist >= blur_rings[0][0]).mean()
print(f"About {reduced_fraction:.0%} of this image is rendered at reduced detail here -- the")
print("book reports VR headsets with real eye tracking saving up to 90% of rendering work.")

The simulation and your own experience share the same logic: invest resolution where it gets
used, predict the rest. But a VR headset knows it is cheating -- the game engine keeps the true
scene in memory and simply chooses not to render all of it. Your brain has no such backup copy.
When it renders your peripheral vision at "low detail plus prediction", there is no ground-truth
image sitting anywhere for it to check against if the prediction is wrong -- which is exactly
why a real object missed at the edge of your vision can go undetected in a way a VR rendering
error never would.

## Pool the class data

One person's peripheral-reading span is noisy. Pool the class: each person adds their mean
chars_left and chars_right (an anonymous ID, not a name) to a shared spreadsheet, downloads it
as CSV and loads it here.

In [ ]:
import io
csv_text = """id,chars_left,chars_right
P01,5.5,6.0
P02,7.0,7.5
P03,4.5,5.0
P04,6.5,7.0
P05,5.0,6.5
"""  # example data: replace with your class CSV, e.g. pd.read_csv("peripheral_reading_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
df["total_span"] = df.chars_left + df.chars_right
boot = [df.total_span.sample(len(df), replace=True).mean() for _ in range(5000)]
lo_ci, hi_ci = np.percentile(boot, [2.5, 97.5])
print(f"Class mean total span: {df.total_span.mean():.1f} characters "
      f"(95% bootstrap interval {lo_ci:.1f}-{hi_ci:.1f}, n = {len(df)})")
df.total_span.plot.hist(bins=6, color="steelblue", alpha=0.7, figsize=(6, 3))
plt.axvspan(8, 16, color="grey", alpha=0.2, label="book's 4-8 per side (8-16 total)")
plt.xlabel("total characters read (left + right)"); plt.legend(); plt.show()

## Questions to think about

1. Your total reading span reflects more than the fovea's fixed 2-degree width -- it also reflects how much your brain can guess from a few clearly-seen letters. What would happen to your measured span if the text were random letters instead of real words?
2. In the foveated-rendering demo, roughly what fraction of the image was rendered at reduced detail? Compare that with the book's claim that "as much as half of V1 is devoted to just the central few degrees of vision" -- are these two numbers measuring the same thing? Why or why not?
3. If a VR headset's foveated-rendering budget rendered sharp detail everywhere the eye had recently looked, not just the current instant, would that better match how your own perception of "everywhere feels sharp" is actually built, or would it be a poor match? Explain.
4. Step 4 asked you to notice when your wiggling fingers become hard to see as you move them outward. How might repeating that test with your fingers moving up and down, rather than side to side, give a different answer, given that cortical magnification is not perfectly symmetric across the visual field?

## Challenge

JPEG compression exploits the same logic as your visual system: it keeps brightness detail
(which you notice) and discards colour detail (which you notice less). Take a photo, save it at
a very low JPEG quality setting, and zoom into both a brightness-rich edge (such as text) and a
smooth colour gradient (such as sky) to see where the compression artefacts appear first.

In [ ]:
# Example data: replace with your own before/after file sizes and quality settings.
my_jpeg_test = pd.DataFrame({
    "quality": [95, 50, 10],                                          # example data
    "file_size_kb": [820, 210, 60],                                    # example data
    "artefacts_first_visible_in": ["none", "text edges", "text and sky both"],  # example data
})
display(my_jpeg_test)
print("If artefacts appear in the smooth colour gradient before they appear in sharp brightness")
print("edges, that matches the book's claim that JPEG 'reduces colour resolution... while")
print("preserving brightness information more carefully' -- the same brightness-over-colour")
print("priority your own retina uses.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-03.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")